In [ ]:
import os
import json
from collections import defaultdict

# ===================== 配置参数 =====================
device = 'results_final'
levels = [3, 4, 5, 6]          # 所有level
resolutions = [256, 512]       # 所有分辨率
models = [
    "qwen2.5-vl-72b",
    "qwen3-vl-32b",
    "qwen3-vl-8b",
    "glm4.6v",
    "conan",
    'qwen3-vl-8b-thinking',
    'qwen3-vl-32b-thinking',
    'qwen3.5-27b',
    'gemini-3.1-pro'
]    # 模型名称
drama_names = [
    "kuang_biao", "huan_le_song", "zhan_chang_sha",
    "yi_qi_tong_guo_chuang_1", "ren_shi_jian", "friends",
    "da_qin_di_guo", "lost", "downton_abbey",
    "san_ti", "chen_mo_de_zhen_xiang", "shan_hai_qing", "zhen_huan_zhuan"
]  # 所有剧集名称
# minibatch数据文件根路径
mini_batch_root = './QA_TEST_DATA_3_API_shuffle_mini_1000_v2'

# ===================== 工具函数 =====================
def get_num_options(options_str):
    """根据选项字符串计算选项个数k"""
    if not isinstance(options_str, str):
        print(f"⚠️  选项格式错误，不是字符串：{options_str}")
        return 0
    options_str = options_str.strip()
    if 'H.' in options_str:
        return 8
    elif 'G.' in options_str:
        return 7
    elif 'F.' in options_str:
        return 6
    elif 'E.' in options_str:
        return 5
    elif 'D.' in options_str:
        return 4
    elif 'C.' in options_str:
        return 3
    elif 'B.' in options_str:
        return 2
    else:
        print(f"❌ 无法识别选项个数，选项内容：{options_str[:50]}...")
        return 0

def load_mini_batch_qa_ids(drama_name):
    """加载指定剧集的minibatch QA ID集合（lv+cid+qid）"""
    mini_batch_file = os.path.join(mini_batch_root, f"{drama_name}.json")
    mini_batch_ids = set()
    
    if not os.path.exists(mini_batch_file):
        print(f"⚠️ minibatch文件不存在：{mini_batch_file}")
        return mini_batch_ids
    
    try:
        with open(mini_batch_file, 'r', encoding='utf-8') as f:
            mini_data = json.load(f)
        
        # 遍历minibatch数据，收集有效ID（lv+cid+qid）
        for lv, cid_dict in mini_data.items():
            for cid, qid_dict in cid_dict.items():
                for qid in qid_dict['Q&A'].keys():
                    # 拼接唯一标识，确保非空
                    if lv and cid and qid:
                        mini_batch_ids.add(f"{lv}_{cid}_{qid}")
        print(f"✅ 加载{drama_name}的minibatch QA数量：{len(mini_batch_ids)}")
    except Exception as e:
        print(f"❌ 读取minibatch文件{mini_batch_file}出错：{str(e)}")
    
    return mini_batch_ids

# ===================== 统计数据结构初始化 =====================
def init_statistics():
    """初始化统计数据结构（新增加权准确率/失败加权/minibatch/QA列表字段）"""
    return {
        # 基础统计
        'total': 0,      # 总问题数
        'correct': 0,    # 答对数量（严格匹配）
        'wrong': 0,      # 答错数量（包含失败）
        'failed': 0,     # result为None/空值的失败数
        # 加权准确率相关
        'correct_weight': 0.0,  # 加权正确数（失败项按1/k计算）
        'accuracy': 0.0,        # 原准确率（失败算错误）
        'accuracy_weight': 0.0, # 加权准确率
        # minibatch统计
        'mini_batch': {
            'total': 0,
            'correct': 0,
            'wrong': 0,
            'failed': 0,
            'correct_weight': 0.0,
            'accuracy': 0.0,
            'accuracy_weight': 0.0
        },
        # QA预测结果列表
        'qa_details': []
    }

def calculate_accuracy(stat):
    """计算准确率（含基础准确率+加权准确率，支持总体/minibatch）"""
    # 计算基础准确率
    if stat['total'] > 0:
        stat['accuracy'] = round(stat['correct'] / stat['total'] * 100, 2)
        stat['accuracy_weight'] = round(stat['correct_weight'] / stat['total'] * 100, 2)
    else:
        stat['accuracy'] = 0.0
        stat['accuracy_weight'] = 0.0
    
    # 计算minibatch准确率
    mini = stat['mini_batch']
    if mini['total'] > 0:
        mini['accuracy'] = round(mini['correct'] / mini['total'] * 100, 2)
        mini['accuracy_weight'] = round(mini['correct_weight'] / mini['total'] * 100, 2)
    else:
        mini['accuracy'] = 0.0
        mini['accuracy_weight'] = 0.0
    
    return stat

# ===================== 核心处理函数 =====================
def process_drama_qa_data(drama_name, level, resolution, model):
    """处理单个剧集的QA数据（新增minibatch/加权准确率/QA详情）"""
    # 1. 确定scale_factor
    scale_factor = '0.1' if drama_name == 'kuang_biao' else '0.2'
    
    # 2. 构建文件路径
    file_path = f'./results.all/L{level}/{resolution}/results.{scale_factor}/qa_data/{drama_name}/{model}/q_a_dict.json'
    
    # 3. 检查文件是否存在
    if not os.path.exists(file_path):
        print(f"⚠️ 文件不存在：{file_path}")
        return None, None
    
    # 4. 加载当前剧集的minibatch QA ID集合
    mini_batch_ids = load_mini_batch_qa_ids(drama_name)
    
    # 5. 初始化统计（总体+维度）
    total_stat = init_statistics()
    dimension_stats = defaultdict(init_statistics)
    
    try:
        # 6. 读取QA数据
        with open(file_path, 'r', encoding='utf-8') as f:
            qa_data = json.load(f)
        
        # 7. 遍历所有QA条目
        for key, content in qa_data.items():
            qa_info = content.get('Q&A', {})
            if not qa_info:
                continue
            
            # 提取核心字段
            answer = qa_info.get('Answer', '').strip()
            result = qa_info.get('result')
            result_str = str(result).strip() if result is not None else ''
            dimension = qa_info.get('Dimension', '未知维度').strip()
            options = qa_info.get('Options', '')
            lv = content.get('level', '').strip()
            cid = content.get('clip_id', '').strip()
            qid = content.get('q_id', '').strip()
            
            # 判断是否在minibatch中
            is_in_mini = False
            
            qa_unique_id = f"{lv}_{cid}_{qid}"
            if qa_unique_id in mini_batch_ids:
                is_in_mini = True
            # 计算选项个数k
            num_option = get_num_options(options)
            
            # 记录QA详情
            qa_detail = {
                'key': key,
                'lv': lv,
                'cid': cid,
                'qid': qid,
                'in_mini_batch': is_in_mini,
                'answer': answer,
                'result': result_str,
                'dimension': dimension,
                'options': options,
                'num_option': num_option,
                'is_correct': False,
                'is_failed': False,
                'correct_weight': 0.0
            }
            
            # 标记是否失败
            is_failed = False
            if result is None or result_str == '':
                is_failed = True
                qa_detail['is_failed'] = True
            
            # ===================== 总体统计 =====================
            total_stat['total'] += 1
            # minibatch统计（如果在mini中）
            if is_in_mini:
                total_stat['mini_batch']['total'] += 1
            
            # 维度统计
            dimension_stats[dimension]['total'] += 1
            if is_in_mini:
                dimension_stats[dimension]['mini_batch']['total'] += 1
            
            # 1. 失败处理
            if is_failed:
                total_stat['failed'] += 1
                total_stat['wrong'] += 1
                if is_in_mini:
                    total_stat['mini_batch']['failed'] += 1
                    total_stat['mini_batch']['wrong'] += 1
                
                dimension_stats[dimension]['failed'] += 1
                dimension_stats[dimension]['wrong'] += 1
                if is_in_mini:
                    dimension_stats[dimension]['mini_batch']['failed'] += 1
                    dimension_stats[dimension]['mini_batch']['wrong'] += 1
                
                # 失败项加权：1/k（k>0时）
                weight = 1.0 / num_option if num_option > 0 else 0.0
                total_stat['correct_weight'] += weight
                if is_in_mini:
                    total_stat['mini_batch']['correct_weight'] += weight
                
                dimension_stats[dimension]['correct_weight'] += weight
                if is_in_mini:
                    dimension_stats[dimension]['mini_batch']['correct_weight'] += weight
                
                qa_detail['correct_weight'] = weight
            
            # 2. 非失败处理（判断是否正确）
            else:
                if answer == result_str:
                    # 严格正确
                    total_stat['correct'] += 1
                    if is_in_mini:
                        total_stat['mini_batch']['correct'] += 1
                    
                    dimension_stats[dimension]['correct'] += 1
                    if is_in_mini:
                        dimension_stats[dimension]['mini_batch']['correct'] += 1
                    
                    # 正确项加权为1
                    total_stat['correct_weight'] += 1.0
                    if is_in_mini:
                        total_stat['mini_batch']['correct_weight'] += 1.0
                    
                    dimension_stats[dimension]['correct_weight'] += 1.0
                    if is_in_mini:
                        dimension_stats[dimension]['mini_batch']['correct_weight'] += 1.0
                    
                    qa_detail['is_correct'] = True
                    qa_detail['correct_weight'] = 1.0
                else:
                    # 答案错误
                    total_stat['wrong'] += 1
                    if is_in_mini:
                        total_stat['mini_batch']['wrong'] += 1
                    
                    dimension_stats[dimension]['wrong'] += 1
                    if is_in_mini:
                        dimension_stats[dimension]['mini_batch']['wrong'] += 1
                    
                    # 错误项加权为0
                    qa_detail['correct_weight'] = 0.0
            
            # 将当前QA详情加入列表
            # total_stat['qa_details'].append(qa_detail)
            dimension_stats[dimension]['qa_details'].append(qa_detail)
        
        # 8. 计算所有准确率
        total_stat = calculate_accuracy(total_stat)
        for dim in dimension_stats:
            dimension_stats[dim] = calculate_accuracy(dimension_stats[dim])
        
        return total_stat, dimension_stats
    
    except Exception as e:
        print(f"❌ 处理文件 {file_path} 出错：{str(e)}")
        return None, None

# ===================== 主函数 =====================
def main():
    """主函数：遍历所有配置并统计"""
    all_results = {}
    
    # 遍历所有配置组合
    for level in levels:
        for resolution in resolutions:
            for model in models:
                for drama_name in drama_names:
                    total_stat, dim_stat = process_drama_qa_data(
                        drama_name, level, resolution, model
                    )
                    
                    if total_stat is None:
                        continue
                    
                    # 存储结果
                    key = f"L{level}_{resolution}_{model}_{drama_name}"
                    all_results[key] = {
                        'total_stat': total_stat,
                        'dimension_stat': dim_stat
                    }
                    
                    # 打印当前配置的统计结果
                    print(f"\n{'='*100}")
                    print(f"统计结果 | Level:{level} | 分辨率:{resolution} | 模型:{model} | 剧集:{drama_name}")
                    print(f"{'='*100}")
                    
                    # 打印总体统计
                    print(f"【总体性能】")
                    print(f"  总问题数：{total_stat['total']}")
                    print(f"  答对数量：{total_stat['correct']} | 答错数量：{total_stat['wrong']}（含失败{total_stat['failed']}）")
                    print(f"  失败数量：{total_stat['failed']}")
                    print(f"  基础准确率：{total_stat['accuracy']}% | 加权准确率：{total_stat['accuracy_weight']}%")
                    
                    # 打印minibatch统计
                    mini = total_stat['mini_batch']
                    print(f"\n【Minibatch性能】")
                    print(f"  总问题数：{mini['total']}")
                    print(f"  答对数量：{mini['correct']} | 答错数量：{mini['wrong']}（含失败{mini['failed']}）")
                    print(f"  失败数量：{mini['failed']}")
                    print(f"  基础准确率：{mini['accuracy']}% | 加权准确率：{mini['accuracy_weight']}%")
                    
                    # 打印维度统计
                    print(f"\n【各维度统计】")
                    for dim, stat in dim_stat.items():
                        print(f"  {dim}:")
                        print(f"    总体：问题数{stat['total']} | 基础准确率{stat['accuracy']}% | 加权准确率{stat['accuracy_weight']}%")
                        print(f"    Mini：问题数{stat['mini_batch']['total']} | 基础准确率{stat['mini_batch']['accuracy']}% | 加权准确率{stat['mini_batch']['accuracy_weight']}%")
    
    # 保存所有结果到JSON文件
    save_path = f'./qa_statistics_results_{device}.json'
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"\n🎉 所有统计完成，结果已保存到：{save_path}")

if __name__ == "__main__":
    main()